<a href="https://colab.research.google.com/github/skayeshaa786o/Internspark-Intership-/blob/main/GUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Cell 1: Install Libraries
# ============================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn plotly ipywidgets

# ============================================================
# Cell 2: Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from google.colab import files
import warning
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print("✅ Libraries imported successfully!")

# ============================================================
# Cell 3: Upload Dataset
# ============================================================
print("📤 Please upload your CSV file...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {filename}")

# ============================================================
# Cell 4: Load & Prepare Data
# ============================================================
df_full = pd.read_csv(filename)
print(f"Full dataset shape: {df_full.shape}")

# Use sample for faster training
df = df_full.sample(n=15000, random_state=42).reset_index(drop=True)
df["ai_system_used"] = df["ai_system_used"].fillna("None")

print(f"Working sample: {df.shape}")
print("\nRisk Category Distribution:")
print(df["risk_category"].value_counts())

# ============================================================
# Cell 5: Preprocessing + Train Models
# ============================================================
RISK_ORDER = ["Low", "Moderate", "High", "Critical"]
le_target = LabelEncoder()
le_target.fit(RISK_ORDER)
y = le_target.transform(df["risk_category"])

cat_cols = ["region", "country", "crop", "farm_type", "soil_type",
            "irrigation_type", "ai_system_used", "technology_stack"]

num_cols = ["year", "farm_size_hectares", "ai_adoption_score", "mechanization_index",
            "iot_sensor_density_per_ha", "drone_usage_hours_per_month", "satellite_monitoring_score",
            "avg_temperature_c", "annual_rainfall_mm", "soil_moisture_percent",
            "fertilizer_kg_per_ha", "pesticide_litre_per_ha", "water_use_m3_per_ha",
            "climate_stress_index", "disease_risk_score", "pest_risk_score",
            "yield_ton_per_ha", "operational_cost_usd_per_ha", "market_price_usd_per_ton",
            "carbon_emission_kgco2e_per_ha", "sustainability_score"]

X = df[cat_cols + num_cols]

# Preprocessor
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=12))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

X_processed = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# Train all models
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel="linear", probability=True, random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=16, random_state=42, n_jobs=-1)
}

print("\nTraining models... Please wait...")
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    results[name] = {"Accuracy": acc, "F1": f1}
    trained_models[name] = model
    print(f"{name:<22} → Accuracy: {acc*100:.2f}%  |  F1: {f1*100:.2f}%")

# Best model
best_name = max(results, key=lambda k: results[k]["F1"])
best_model = trained_models[best_name]
print(f"\n🏆 Best Model: {best_name} (F1 = {results[best_name]['F1']*100:.2f}%)")

# ============================================================
# Cell 6: Interactive GUI
# ============================================================
print("\n" + "="*60)
print("🎮 INTERACTIVE PREDICTION GUI")
print("="*60)

# Create widgets
region_w = widgets.Dropdown(options=sorted(df_full["region"].dropna().unique()), description="Region:", style={'description_width': '120px'})
crop_w = widgets.Dropdown(options=sorted(df_full["crop"].dropna().unique()), description="Crop:", style={'description_width': '120px'})
farm_type_w = widgets.Dropdown(options=sorted(df_full["farm_type"].dropna().unique()), description="Farm Type:", style={'description_width': '120px'})
soil_w = widgets.Dropdown(options=sorted(df_full["soil_type"].dropna().unique()), description="Soil Type:", style={'description_width': '120px'})
irrigation_w = widgets.Dropdown(options=sorted(df_full["irrigation_type"].dropna().unique()), description="Irrigation:", style={'description_width': '120px'})
ai_system_w = widgets.Dropdown(options=sorted(df_full["ai_system_used"].fillna("None").unique()), description="AI System:", style={'description_width': '120px'})
tech_w = widgets.Dropdown(options=sorted(df_full["technology_stack"].dropna().unique()), description="Tech Stack:", style={'description_width': '120px'})

year_w = widgets.IntSlider(value=2035, min=2020, max=2050, description="Year:", style={'description_width': '120px'})
farm_size_w = widgets.FloatSlider(value=25.0, min=1, max=150, step=0.5, description="Farm Size (ha):", style={'description_width': '120px'})
ai_score_w = widgets.FloatSlider(value=60.0, min=0, max=100, description="AI Adoption:", style={'description_width': '120px'})
climate_w = widgets.FloatSlider(value=40.0, min=10, max=100, description="Climate Stress:", style={'description_width': '120px'})
disease_w = widgets.FloatSlider(value=30.0, min=5, max=70, description="Disease Risk:", style={'description_width': '120px'})
pest_w = widgets.FloatSlider(value=50.0, min=10, max=100, description="Pest Risk:", style={'description_width': '120px'})
temp_w = widgets.FloatSlider(value=27.0, min=15, max=42, description="Temperature °C:", style={'description_width': '120px'})
rain_w = widgets.FloatSlider(value=600.0, min=50, max=1500, description="Rainfall (mm):", style={'description_width': '120px'})
yield_w = widgets.FloatSlider(value=15.0, min=1, max=80, description="Yield (t/ha):", style={'description_width': '120px'})

predict_btn = widgets.Button(
    description="🔮 Predict Risk Category",
    button_style="success",
    layout=widgets.Layout(width="300px", height="45px")
)
output = widgets.Output()

def on_predict_clicked(b):
    with output:
        clear_output()

        input_data = {
            "region": region_w.value,
            "country": "Unknown",
            "crop": crop_w.value,
            "farm_type": farm_type_w.value,
            "soil_type": soil_w.value,
            "irrigation_type": irrigation_w.value,
            "ai_system_used": ai_system_w.value,
            "technology_stack": tech_w.value,
            "year": year_w.value,
            "farm_size_hectares": farm_size_w.value,
            "ai_adoption_score": ai_score_w.value,
            "mechanization_index": 55.0,
            "iot_sensor_density_per_ha": 8.0,
            "drone_usage_hours_per_month": 5.0,
            "satellite_monitoring_score": 50.0,
            "avg_temperature_c": temp_w.value,
            "annual_rainfall_mm": rain_w.value,
            "soil_moisture_percent": 35.0,
            "fertilizer_kg_per_ha": 120.0,
            "pesticide_litre_per_ha": 2.0,
            "water_use_m3_per_ha": 600.0,
            "climate_stress_index": climate_w.value,
            "disease_risk_score": disease_w.value,
            "pest_risk_score": pest_w.value,
            "yield_ton_per_ha": yield_w.value,
            "operational_cost_usd_per_ha": 1100.0,
            "market_price_usd_per_ton": 350.0,
            "carbon_emission_kgco2e_per_ha": 900.0,
            "sustainability_score": 50.0
        }

        input_df = pd.DataFrame([input_data])
        X_in = preprocessor.transform(input_df[cat_cols + num_cols])

        pred_idx = best_model.predict(X_in)[0]
        proba = best_model.predict_proba(X_in)[0]
        pred_label = RISK_ORDER[pred_idx]

        # Color mapping
        colors = {"Low": "#28a745", "Moderate": "#ffc107", "High": "#fd7e14", "Critical": "#dc3545"}

        print("=" * 55)
        print(f"🌾 PREDICTED RISK CATEGORY: {pred_label}")
        print("=" * 55)
        print(f"\nUsing Best Model: {best_name}")
        print("\nClass Probabilities:")
        for label, p in zip(RISK_ORDER, proba):
            bar = "█" * int(p * 30)
            print(f"  {label:12s}: {p:.3f}  {bar}")

        recommendations = {
            "Low": "✅ Maintain current smart farming plan. Continue regular monitoring.",
            "Moderate": "⚠️ Deploy pest monitoring traps + precision spraying. Consider IoT soil sensors.",
            "High": "🚨 Adopt climate-resilient variety + heat-stress advisory. Shift to smart/drip irrigation.",
            "Critical": "🔴 URGENT: Full precision agriculture stack + climate-resilient variety required immediately."
        }
        print(f"\n💡 Recommended Action:\n   {recommendations[pred_label]}")
        print("=" * 55)

predict_btn.on_click(on_predict_clicked)

# Display GUI
gui = widgets.VBox([
    widgets.HTML("<h2 style='color:#1B5E20;'>🌾 AgriAI - Crop Risk Prediction System</h2>"),
    widgets.HTML(f"<b>Best Model:</b> {best_name} &nbsp;&nbsp;|&nbsp;&nbsp; <b>Accuracy:</b> {results[best_name]['Accuracy']*100:.2f}%"),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>📋 Select Farm Details:</b>"),
    widgets.HBox([region_w, crop_w]),
    widgets.HBox([farm_type_w, soil_w]),
    widgets.HBox([irrigation_w, ai_system_w]),
    tech_w,
    widgets.HTML("<br><b>📊 Adjust Numeric Values:</b>"),
    year_w, farm_size_w, ai_score_w,
    climate_w, disease_w, pest_w,
    temp_w, rain_w, yield_w,
    widgets.HTML("<br>"),
    predict_btn,
    output
])

display(gui)

print("\n✅ GUI is ready! Adjust the values and click 'Predict Risk Category'")


✅ GUI is ready! Adjust the values and click 'Predict Risk Category'


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
print("Widgets enabled!")

Widgets enabled!


In [ ]:
# ============================================================
# Cell 1: Install Libraries
# ============================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn plotly ipywidgets

# ============================================================
# Cell 2: Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from google.colab import files
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print("✅ Libraries imported successfully!")

# ============================================================
# Cell 3: Upload Dataset
# ============================================================
print("📤 Please upload your CSV file...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {filename}")

# ============================================================
# Cell 4: Load & Prepare Data
# ============================================================
df_full = pd.read_csv(filename)
print(f"Full dataset shape: {df_full.shape}")

# Use sample for faster training
df = df_full.sample(n=15000, random_state=42).reset_index(drop=True)
df["ai_system_used"] = df["ai_system_used"].fillna("None")

print(f"Working sample: {df.shape}")
print("\nRisk Category Distribution:")
print(df["risk_category"].value_counts())

# ============================================================
# Cell 5: Preprocessing + Train Models
# ============================================================
RISK_ORDER = ["Low", "Moderate", "High", "Critical"]
le_target = LabelEncoder()
le_target.fit(RISK_ORDER)
y = le_target.transform(df["risk_category"])

cat_cols = ["region", "country", "crop", "farm_type", "soil_type",
            "irrigation_type", "ai_system_used", "technology_stack"]

num_cols = ["year", "farm_size_hectares", "ai_adoption_score", "mechanization_index",
            "iot_sensor_density_per_ha", "drone_usage_hours_per_month", "satellite_monitoring_score",
            "avg_temperature_c", "annual_rainfall_mm", "soil_moisture_percent",
            "fertilizer_kg_per_ha", "pesticide_litre_per_ha", "water_use_m3_per_ha",
            "climate_stress_index", "disease_risk_score", "pest_risk_score",
            "yield_ton_per_ha", "operational_cost_usd_per_ha", "market_price_usd_per_ton",
            "carbon_emission_kgco2e_per_ha", "sustainability_score"]

X = df[cat_cols + num_cols]

# Preprocessor
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=12))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

X_processed = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# Train all models
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel="linear", probability=True, random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=16, random_state=42, n_jobs=-1)
}

print("\nTraining models... Please wait...")
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    results[name] = {"Accuracy": acc, "F1": f1}
    trained_models[name] = model
    print(f"{name:<22} → Accuracy: {acc*100:.2f}%  |  F1: {f1*100:.2f}%")

# Best model
best_name = max(results, key=lambda k: results[k]["F1"])
best_model = trained_models[best_name]
print(f"\n🏆 Best Model: {best_name} (F1 = {results[best_name]['F1']*100:.2f}%)")

# ============================================================
# Cell 6: Interactive GUI
# ============================================================
print("\n" + "="*60)
print("🎮 INTERACTIVE PREDICTION GUI")
print("="*60)

# Create widgets
region_w = widgets.Dropdown(options=sorted(df_full["region"].dropna().unique()), description="Region:", style={'description_width': '120px'})
crop_w = widgets.Dropdown(options=sorted(df_full["crop"].dropna().unique()), description="Crop:", style={'description_width': '120px'})
farm_type_w = widgets.Dropdown(options=sorted(df_full["farm_type"].dropna().unique()), description="Farm Type:", style={'description_width': '120px'})
soil_w = widgets.Dropdown(options=sorted(df_full["soil_type"].dropna().unique()), description="Soil Type:", style={'description_width': '120px'})
irrigation_w = widgets.Dropdown(options=sorted(df_full["irrigation_type"].dropna().unique()), description="Irrigation:", style={'description_width': '120px'})
ai_system_w = widgets.Dropdown(options=sorted(df_full["ai_system_used"].fillna("None").unique()), description="AI System:", style={'description_width': '120px'})
tech_w = widgets.Dropdown(options=sorted(df_full["technology_stack"].dropna().unique()), description="Tech Stack:", style={'description_width': '120px'})

year_w = widgets.IntSlider(value=2035, min=2020, max=2050, description="Year:", style={'description_width': '120px'})
farm_size_w = widgets.FloatSlider(value=25.0, min=1, max=150, step=0.5, description="Farm Size (ha):", style={'description_width': '120px'})
ai_score_w = widgets.FloatSlider(value=60.0, min=0, max=100, description="AI Adoption:", style={'description_width': '120px'})
climate_w = widgets.FloatSlider(value=40.0, min=10, max=100, description="Climate Stress:", style={'description_width': '120px'})
disease_w = widgets.FloatSlider(value=30.0, min=5, max=70, description="Disease Risk:", style={'description_width': '120px'})
pest_w = widgets.FloatSlider(value=50.0, min=10, max=100, description="Pest Risk:", style={'description_width': '120px'})
temp_w = widgets.FloatSlider(value=27.0, min=15, max=42, description="Temperature °C:", style={'description_width': '120px'})
rain_w = widgets.FloatSlider(value=600.0, min=50, max=1500, description="Rainfall (mm):", style={'description_width': '120px'})
yield_w = widgets.FloatSlider(value=15.0, min=1, max=80, description="Yield (t/ha):", style={'description_width': '120px'})

predict_btn = widgets.Button(
    description="🔮 Predict Risk Category",
    button_style="success",
    layout=widgets.Layout(width="300px", height="45px")
)
output = widgets.Output()

def on_predict_clicked(b):
    with output:
        clear_output()

        input_data = {
            "region": region_w.value,
            "country": "Unknown",
            "crop": crop_w.value,
            "farm_type": farm_type_w.value,
            "soil_type": soil_w.value,
            "irrigation_type": irrigation_w.value,
            "ai_system_used": ai_system_w.value,
            "technology_stack": tech_w.value,
            "year": year_w.value,
            "farm_size_hectares": farm_size_w.value,
            "ai_adoption_score": ai_score_w.value,
            "mechanization_index": 55.0,
            "iot_sensor_density_per_ha": 8.0,
            "drone_usage_hours_per_month": 5.0,
            "satellite_monitoring_score": 50.0,
            "avg_temperature_c": temp_w.value,
            "annual_rainfall_mm": rain_w.value,
            "soil_moisture_percent": 35.0,
            "fertilizer_kg_per_ha": 120.0,
            "pesticide_litre_per_ha": 2.0,
            "water_use_m3_per_ha": 600.0,
            "climate_stress_index": climate_w.value,
            "disease_risk_score": disease_w.value,
            "pest_risk_score": pest_w.value,
            "yield_ton_per_ha": yield_w.value,
            "operational_cost_usd_per_ha": 1100.0,
            "market_price_usd_per_ton": 350.0,
            "carbon_emission_kgco2e_per_ha": 900.0,
            "sustainability_score": 50.0
        }

        input_df = pd.DataFrame([input_data])
        X_in = preprocessor.transform(input_df[cat_cols + num_cols])

        pred_idx = best_model.predict(X_in)[0]
        proba = best_model.predict_proba(X_in)[0]
        pred_label = RISK_ORDER[pred_idx]

        # Color mapping
        colors = {"Low": "#28a745", "Moderate": "#ffc107", "High": "#fd7e14", "Critical": "#dc3545"}

        print("=" * 55)
        print(f"🌾 PREDICTED RISK CATEGORY: {pred_label}")
        print("=" * 55)
        print(f"\nUsing Best Model: {best_name}")
        print("\nClass Probabilities:")
        for label, p in zip(RISK_ORDER, proba):
            bar = "█" * int(p * 30)
            print(f"  {label:12s}: {p:.3f}  {bar}")

        recommendations = {
            "Low": "✅ Maintain current smart farming plan. Continue regular monitoring.",
            "Moderate": "⚠️ Deploy pest monitoring traps + precision spraying. Consider IoT soil sensors.",
            "High": "🚨 Adopt climate-resilient variety + heat-stress advisory. Shift to smart/drip irrigation.",
            "Critical": "🔴 URGENT: Full precision agriculture stack + climate-resilient variety required immediately."
        }
        print(f"\n💡 Recommended Action:\n   {recommendations[pred_label]}")
        print("=" * 55)

predict_btn.on_click(on_predict_clicked)

# Display GUI
gui = widgets.VBox([
    widgets.HTML("<h2 style='color:#1B5E20;'>🌾 AgriAI - Crop Risk Prediction System</h2>"),
    widgets.HTML(f"<b>Best Model:</b> {best_name} &nbsp;&nbsp;|&nbsp;&nbsp; <b>Accuracy:</b> {results[best_name]['Accuracy']*100:.2f}%"),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>📋 Select Farm Details:</b>"),
    widgets.HBox([region_w, crop_w]),
    widgets.HBox([farm_type_w, soil_w]),
    widgets.HBox([irrigation_w, ai_system_w]),
    tech_w,
    widgets.HTML("<br><b>📊 Adjust Numeric Values:</b>"),
    year_w, farm_size_w, ai_score_w,
    climate_w, disease_w, pest_w,
    temp_w, rain_w, yield_w,
    widgets.HTML("<br>"),
    predict_btn,
    output
])

display(gui)

print("\n✅ GUI is ready! Adjust the values and click 'Predict Risk Category'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 13.9 MB/s eta 0:00:00
✅ Libraries imported successfully!
📤 Please upload your CSV file...


Saving modern_agriculture_ai_crop_production_2020_2050.csv to modern_agriculture_ai_crop_production_2020_2050.csv
✅ Uploaded: modern_agriculture_ai_crop_production_2020_2050.csv
Full dataset shape: (75000, 36)
Working sample: (15000, 36)

Risk Category Distribution:
risk_category
Moderate    11825
High         2520
Low           636
Critical       19
Name: count, dtype: int64

Training models... Please wait...
Decision Tree          → Accuracy: 95.43%  |  F1: 95.38%
Naive Bayes            → Accuracy: 46.03%  |  F1: 58.63%
Logistic Regression    → Accuracy: 98.77%  |  F1: 98.75%
SVM                    → Accuracy: 98.57%  |  F1: 98.59%
Random Forest          → Accuracy: 90.77%  |  F1: 88.99%

🏆 Best Model: Logistic Regression (F1 = 98.75%)

🎮 INTERACTIVE PREDICTION GUI



✅ GUI is ready! Adjust the values and click 'Predict Risk Category'
